In [ ]:
!pip install -q datasets sentence-transformers pandas numpy tqdm scikit-learn transformers sentencepiece sacremoses

In [ ]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm
import warnings
import torch

warnings.filterwarnings('ignore')

device = "cuda"

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
SEED = 42

In [ ]:
dataset = load_dataset("ailsntua/QEvasion")
raw_train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

train_df, val_df = train_test_split(
    raw_train_df,
    test_size=700,
    random_state=SEED,
    stratify=raw_train_df['clarity_label']
)

val_indices = set(val_df.index.tolist())

In [ ]:
TRANSLATION_METHODS = [
    'zh',
    'es->ru',
]

translation_models = {}

def load_translation_model(src_lang: str, tgt_lang: str):
    model_name = f"Helsinki-NLP/opus-mt-{src_lang}-{tgt_lang}"
    cache_key = f"{src_lang}-{tgt_lang}"

    if cache_key not in translation_models:
        try:
            tokenizer = MarianTokenizer.from_pretrained(model_name)
            model = MarianMTModel.from_pretrained(model_name).to(device)
            model.eval()
            translation_models[cache_key] = (tokenizer, model)
        except Exception as e:
            print(f"Failed to load {model_name}: {e}")
            return None

    return translation_models.get(cache_key)


def translate(text: str, src_lang: str, tgt_lang: str, max_length: int = 512) -> str:
    result = load_translation_model(src_lang, tgt_lang)
    if result is None:
        return None

    tokenizer, model = result
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, num_beams=4, early_stopping=True)

    translated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translated


def back_translate_single_hop(text: str, pivot_lang: str) -> str:
    intermediate = translate(text, 'en', pivot_lang)
    if intermediate is None:
        return None

    back_translated = translate(intermediate, pivot_lang, 'en')
    return back_translated


def back_translate_multi_hop(text: str, lang_chain: list) -> str:
    current_text = text
    prev_lang = 'en'

    for lang in lang_chain:
        current_text = translate(current_text, prev_lang, lang)
        if current_text is None:
            return None
        prev_lang = lang

    final_text = translate(current_text, prev_lang, 'en')
    return final_text


def back_translate(text: str, method: str) -> tuple:
    if '->' in method:
        langs = method.split('->')
        result = back_translate_multi_hop(text, langs)
        return result, f"en->{method}->en"
    else:
        result = back_translate_single_hop(text, method)
        return result, f"en->{method}->en"

In [ ]:
required_models = [
    ('en', 'zh'),
    ('zh', 'en'),
    ('en', 'es'),
    ('es', 'ru'),
    ('ru', 'en'),
]

all_loaded = True
for src, tgt in required_models:
    result = load_translation_model(src, tgt)
    if result:
        continue
    else:
        all_loaded = False

if not all_loaded:
    print("Failed loading some models")

In [ ]:
MIN_SIMILARITY = 0.7
MAX_SIMILARITY = 0.95

def compute_similarity(original_text: str, augmented_text: str) -> float:
    embeddings = embedding_model.encode([original_text, augmented_text])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return similarity


def is_valid_augmentation(
    original_q: str,
    original_a: str,
    augmented_q: str,
    augmented_a: str,
    min_similarity: float = MIN_SIMILARITY,
    max_similarity: float = MAX_SIMILARITY
) -> tuple:

    q_sim = compute_similarity(original_q, augmented_q)
    a_sim = compute_similarity(original_a, augmented_a)

    original_combined = f"Question: {original_q} Answer: {original_a}"
    augmented_combined = f"Question: {augmented_q} Answer: {augmented_a}"
    combined_sim = compute_similarity(original_combined, augmented_combined)

    is_valid = min_similarity <= combined_sim <= max_similarity

    return is_valid, q_sim, a_sim, combined_sim

In [ ]:
def back_translate_qa_pair(
    question: str,
    answer: str,
    method: str
) -> dict:
    try:
        bt_question, q_method = back_translate(question, method)
        if bt_question is None:
            return None

        bt_answer, a_method = back_translate(answer, method)
        if bt_answer is None:
            return None

        return {
            'question': bt_question,
            'answer': bt_answer,
            'method': method,
            'method_description': q_method
        }
    except Exception as e:
        print(f"Back-translation error ({method}): {e}")
        return None

In [ ]:
def augment_minority_classes_backtranslation(
    train_df: pd.DataFrame,
    targets: dict,
    methods: list = TRANSLATION_METHODS,
    min_similarity: float = MIN_SIMILARITY,
    max_similarity: float = MAX_SIMILARITY
) -> pd.DataFrame:

    augmented_samples = []

    for label, to_generate in targets.items():

        if to_generate <= 0:
            continue

        class_samples = train_df[train_df['clarity_label'] == label].copy()

        sample_idx = 0
        method_idx = 0
        generated = 0

        pbar = tqdm(total=to_generate, desc=f"Generating {label}")

        while generated < to_generate:
            original = class_samples.iloc[sample_idx % len(class_samples)]
            method = methods[method_idx % len(methods)]
            method_idx += 1
            if method_idx % len(methods) == 0:
                sample_idx += 1

            result = back_translate_qa_pair(
                original['question'],
                original['interview_answer'],
                method
            )

            if result is None:
                continue

            is_valid, q_sim, a_sim, combined_sim = is_valid_augmentation(
                original['question'],
                original['interview_answer'],
                result['question'],
                result['answer'],
                min_similarity=min_similarity,
                max_similarity=max_similarity
            )

            method_type = 'multi-hop' if '->' in method else 'single-hop'

            if is_valid:
                augmented_samples.append({
                    'question': result['question'],
                    'interview_answer': result['answer'],
                    'clarity_label': label,
                    'is_augmented': True,
                    'augmentation_method': 'back_translation',
                    'translation_method': method,
                    'method_type': method_type,
                    'original_idx': original.name,
                    'q_similarity': q_sim,
                    'a_similarity': a_sim,
                    'combined_similarity': combined_sim
                })
                generated += 1
                pbar.update(1)

            if sample_idx > len(class_samples) * 5 and generated < to_generate:
                print(f"\nWarning: Struggled to generate valid samples for {label}")
                break

        pbar.close()

    return pd.DataFrame(augmented_samples)

In [ ]:
augmented_df = augment_minority_classes_backtranslation(
    train_df=train_df,
    targets=augmentation_targets,
    methods=TRANSLATION_METHODS,
    min_similarity=MIN_SIMILARITY,
    max_similarity=MAX_SIMILARITY,
)

In [ ]:
original_train = train_df.copy()
original_train['is_augmented'] = False
original_train['augmentation_method'] = 'original'
original_train['translation_method'] = None
original_train['method_type'] = None
original_train['original_idx'] = original_train.index
original_train['q_similarity'] = 1.0
original_train['a_similarity'] = 1.0
original_train['combined_similarity'] = 1.0

columns_to_keep = ['question', 'interview_answer', 'clarity_label', 'is_augmented',
                    'augmentation_method', 'translation_method', 'method_type',
                    'original_idx', 'q_similarity', 'a_similarity', 'combined_similarity']

combined_train_df = pd.concat([
    original_train[columns_to_keep],
    augmented_df[columns_to_keep]
], ignore_index=True)

In [ ]:
import os
os.makedirs('../data', exist_ok=True)
combined_train_df.to_csv('../data/train_augmented_backtranslation.csv', index=False)
val_df.to_csv('../data/validation_clean.csv', index=False)
augmented_df.to_json('../data/augmented_samples_backtranslation.json', orient='records', indent=2)


In [ ]:
original_dataset = load_dataset("ailsntua/QEvasion")
test_df = original_dataset['test'].to_pandas()

essential_columns = ['question', 'interview_answer', 'clarity_label', 'is_augmented']

train_for_hf = combined_train_df[['question', 'interview_answer', 'clarity_label', 'is_augmented']].copy()

val_for_hf = val_df[['question', 'interview_answer', 'clarity_label']].copy()
val_for_hf['is_augmented'] = False

test_for_hf = test_df[['question', 'interview_answer', 'clarity_label']].copy()
test_for_hf['is_augmented'] = False

train_dataset = Dataset.from_pandas(train_for_hf, preserve_index=False)
val_dataset = Dataset.from_pandas(val_for_hf, preserve_index=False)
test_dataset = Dataset.from_pandas(test_for_hf, preserve_index=False)

augmented_dataset = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

In [ ]:
from huggingface_hub import login
login()
augmented_dataset.push_to_hub("gabrielstefan04/qevasion-backtranslation-augmented")